# SolarShield — Notebook 01: Data Preparation
**Project**: SolarShield — ML-Driven Predictive Maintenance & Efficiency Analysis  
**College**: St Joseph Engineering College, Mangaluru (VTU)  

**Purpose**: Load Kaggle 'Solar Power Generation Data' by anikannal, merge generation + weather,  
engineer the 12 rolling features, synthesize labels, and save `data/solar_features.csv`.

In [13]:
import pandas as pd
import numpy as np
import os

# ── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
ASSUMED_VOLTAGE = 24.0   # V  (used when voltage column is absent)
RATED_POWER     = 100.0  # W  (adjust to actual panel spec if known)
WINDOW_MINUTES  = 30
READINGS_PER_MINUTE = 1 / 10  # ESP32 sends every 10 min → 3 readings / 30-min window

os.makedirs('data', exist_ok=True)
os.makedirs('ml_models', exist_ok=True)

print('Dependencies loaded.')

Dependencies loaded.


## 1. Load raw CSVs
Download from Kaggle (anikannal / solar-power-generation-data):
- `Plant_1_Generation_Data.csv`
- `Plant_1_Weather_Sensor_Data.csv`

Place both files in the `data/` directory, then run the cells below.

In [14]:
gen_path     = '../data/Plant_1_Generation_Data.csv'
weather_path = '../data/Plant_1_Weather_Sensor_Data.csv'

gen_df     = pd.read_csv(gen_path)
weather_df = pd.read_csv(weather_path)

print('Generation shape :', gen_df.shape)
print('Weather shape    :', weather_df.shape)
print('\nGeneration columns:'); print(gen_df.columns.tolist())
print('\nWeather columns:'  ); print(weather_df.columns.tolist())

Generation shape : (5000, 3)
Weather shape    : (5000, 4)

Generation columns:
['DATE_TIME', 'DC_POWER', 'AC_POWER']

Weather columns:
['DATE_TIME', 'IRRADIATION', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE']


## 2. Parse timestamps and merge

In [15]:
# Normalise timestamp column name (dataset uses 'DATE_TIME')
for df, name in [(gen_df, 'gen_df'), (weather_df, 'weather_df')]:
    ts_col = next((c for c in df.columns if 'DATE' in c.upper() or 'TIME' in c.upper()), None)
    if ts_col is None:
        raise ValueError(f'No timestamp column found in {name}. Columns: {df.columns.tolist()}')
    df.rename(columns={ts_col: 'DATE_TIME'}, inplace=True)
    df['DATE_TIME'] = pd.to_datetime(df['DATE_TIME'], dayfirst=True, errors='coerce')

# Drop rows with unparseable timestamps
gen_df.dropna(subset=['DATE_TIME'], inplace=True)
weather_df.dropna(subset=['DATE_TIME'], inplace=True)

# Merge on nearest timestamp (tolerance = 5 min) to handle slight offsets
gen_df     = gen_df.sort_values('DATE_TIME').reset_index(drop=True)
weather_df = weather_df.sort_values('DATE_TIME').reset_index(drop=True)

df = pd.merge_asof(
    gen_df, weather_df,
    on='DATE_TIME',
    tolerance=pd.Timedelta('5min'),
    direction='nearest'
)

print(f'Merged shape: {df.shape}')
df.head(3)

Merged shape: (2304, 6)


,DATE_TIME,DC_POWER,AC_POWER,IRRADIATION,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE
0,2022-01-01 00:00:00,0.0,0.0,0.0,22.95,29.14
1,2022-01-01 00:15:00,0.0,0.0,0.0,21.85,25.60
2,2022-01-01 00:30:00,0.0,0.0,0.0,23.07,29.89


## 3. Build raw sensor columns
The Kaggle dataset does **not** include separate voltage/current readings.  
We derive them from `DC_POWER` using an assumed bus voltage of 24 V.

In [16]:
# ── Power ────────────────────────────────────────────────────────────────────
if 'DC_POWER' in df.columns:
    df['power'] = df['DC_POWER'].clip(lower=0)
elif 'AC_POWER' in df.columns:
    df['power'] = df['AC_POWER'].clip(lower=0)
else:
    raise ValueError('Neither DC_POWER nor AC_POWER found. Check CSV.')

# ── Voltage / current (derived when absent) ──────────────────────────────────
if 'VOLTAGE' in df.columns:
    df['voltage'] = df['VOLTAGE'].clip(lower=0)
else:
    df['voltage'] = ASSUMED_VOLTAGE  # constant assumption

df['current'] = df['power'] / (df['voltage'].replace(0, ASSUMED_VOLTAGE))

# ── Irradiance (lux proxy) ────────────────────────────────────────────────────
irrad_col = next((c for c in df.columns if 'IRRADIATION' in c.upper() or 'IRR' in c.upper()), None)
if irrad_col:
    # Kaggle irradiation is in W/m²; convert to lux (rough: 1 W/m² ≈ 120 lux)
    df['lux'] = (df[irrad_col] * 120).clip(lower=0)
else:
    # Fallback: scale ambient temp as noise
    df['lux'] = 10000.0
    print('WARNING: No irradiation column found; using constant lux=10000.')

# ── Temperature & humidity ────────────────────────────────────────────────────
temp_col = next((c for c in df.columns if 'TEMP' in c.upper() or 'AMBIENT' in c.upper()), None)
hum_col  = next((c for c in df.columns if 'HUMID' in c.upper()), None)

df['temperature'] = df[temp_col].clip(-10, 80) if temp_col else 28.0
df['humidity']    = df[hum_col].clip(0, 100)   if hum_col  else 65.0

if not temp_col: print('WARNING: No temperature column found; using constant 28°C.')
if not hum_col:  print('WARNING: No humidity column found; using constant 65%.')

print('Raw columns built. Sample:')
df[['DATE_TIME','power','voltage','current','lux','temperature','humidity']].head()

Raw columns built. Sample:


,DATE_TIME,power,voltage,current,lux,temperature,humidity
0,2022-01-01 00:00:00,0.0,24.0,0.0,0.0,22.95,65.0
1,2022-01-01 00:15:00,0.0,24.0,0.0,0.0,21.85,65.0
2,2022-01-01 00:30:00,0.0,24.0,0.0,0.0,23.07,65.0
3,2022-01-01 00:45:00,0.0,24.0,0.0,0.0,22.28,65.0
4,2022-01-01 01:00:00,0.0,24.0,0.0,0.0,22.55,65.0


## 4. Rolling 30-minute feature engineering

In [17]:
df = df.sort_values('DATE_TIME').reset_index(drop=True)

# Rolling window: dataset cadence = 15 min → 2 rows per 30-min window
# Detect cadence automatically
cadence_sec = df['DATE_TIME'].diff().median().total_seconds()
window_rows = max(2, int((WINDOW_MINUTES * 60) / cadence_sec))
print(f'Detected cadence: {cadence_sec/60:.1f} min → rolling window = {window_rows} rows')

def rolling_stat(series, window, stat):
    if stat == 'mean': return series.rolling(window, min_periods=1).mean()
    if stat == 'std':  return series.rolling(window, min_periods=1).std().fillna(0)

feat = pd.DataFrame()
feat['DATE_TIME']   = df['DATE_TIME']
feat['voltage_mean'] = rolling_stat(df['voltage'], window_rows, 'mean')
feat['voltage_std']  = rolling_stat(df['voltage'], window_rows, 'std')
feat['current_mean'] = rolling_stat(df['current'], window_rows, 'mean')
feat['current_std']  = rolling_stat(df['current'], window_rows, 'std')
feat['power_mean']   = rolling_stat(df['power'],   window_rows, 'mean')
feat['power_std']    = rolling_stat(df['power'],   window_rows, 'std')
feat['lux_mean']     = rolling_stat(df['lux'],     window_rows, 'mean')
feat['temperature_mean'] = rolling_stat(df['temperature'], window_rows, 'mean')
feat['humidity_mean']    = rolling_stat(df['humidity'],    window_rows, 'mean')

# Rate-of-change features: (last - first) / elapsed_minutes over the window
elapsed_min = (cadence_sec * (window_rows - 1)) / 60
elapsed_min = max(elapsed_min, 1)  # avoid division by zero

feat['power_rate_of_change']   = df['power'].diff(window_rows - 1).fillna(0) / elapsed_min
feat['voltage_rate_of_change'] = df['voltage'].diff(window_rows - 1).fillna(0) / elapsed_min

# Efficiency ratio: power_mean / (lux_mean / 1000 + 0.001)
feat['efficiency_ratio'] = feat['power_mean'] / (feat['lux_mean'] / 1000 + 0.001)

print('Feature engineering complete. Shape:', feat.shape)
feat.head()

Detected cadence: 15.0 min → rolling window = 2 rows
Feature engineering complete. Shape: (2304, 13)


,DATE_TIME,voltage_mean,voltage_std,current_mean,current_std,power_mean,power_std,lux_mean,temperature_mean,humidity_mean,power_rate_of_change,voltage_rate_of_change,efficiency_ratio
0,2022-01-01 00:00:00,24.0,0.0,0.0,0.0,0.0,0.0,0.0,22.950,65.0,0.0,0.0,0.0
1,2022-01-01 00:15:00,24.0,0.0,0.0,0.0,0.0,0.0,0.0,22.400,65.0,0.0,0.0,0.0
2,2022-01-01 00:30:00,24.0,0.0,0.0,0.0,0.0,0.0,0.0,22.460,65.0,0.0,0.0,0.0
3,2022-01-01 00:45:00,24.0,0.0,0.0,0.0,0.0,0.0,0.0,22.675,65.0,0.0,0.0,0.0
4,2022-01-01 01:00:00,24.0,0.0,0.0,0.0,0.0,0.0,0.0,22.415,65.0,0.0,0.0,0.0


## 5. Synthesise labels

In [18]:
# ── fault_class ───────────────────────────────────────────────────────────────
#   0 = Normal, 1 = Degraded, 2 = Fault
def assign_fault_class(er):
    if er < 1.5:  return 2
    if er < 3.0:  return 1
    return 0

feat['fault_class'] = feat['efficiency_ratio'].apply(assign_fault_class)

# ── efficiency_score ─────────────────────────────────────────────────────────
feat['efficiency_score'] = (df['power'] / RATED_POWER * 100).clip(0, 100).values

# ── maintenance_days ─────────────────────────────────────────────────────────
#   Simulates a 90-day cleaning cycle; resets every 144 rows (≈ 1 day @ 10-min)
feat['maintenance_days'] = feat.index.map(
    lambda i: max(0, min(365, 90 - (i // 144)))
)

print('Label distribution (fault_class):')
print(feat['fault_class'].value_counts().sort_index())
print('\nefficiency_score stats:'); print(feat['efficiency_score'].describe())
print('\nmaintenance_days stats:'); print(feat['maintenance_days'].describe())

Label distribution (fault_class):
fault_class
0    1147
2    1157
Name: count, dtype: int64

efficiency_score stats:
count    2304.000000
mean       31.518186
std        38.219595
min         0.000000
25%         0.000000
50%         0.000000
75%        69.420000
max       100.000000
Name: efficiency_score, dtype: float64

maintenance_days stats:
count    2304.000000
mean       82.500000
std         4.610773
min        75.000000
25%        78.750000
50%        82.500000
75%        86.250000
max        90.000000
Name: maintenance_days, dtype: float64


## 6. Finalise and save

In [19]:
FEATURE_ORDER = [
    'voltage_mean', 'voltage_std',
    'current_mean', 'current_std',
    'power_mean',   'power_std',
    'lux_mean',
    'temperature_mean', 'humidity_mean',
    'power_rate_of_change', 'voltage_rate_of_change',
    'efficiency_ratio'
]
TARGET_COLS = ['fault_class', 'efficiency_score', 'maintenance_days']

output_cols = FEATURE_ORDER + TARGET_COLS
final_df = feat[output_cols].dropna().reset_index(drop=True)

print(f'Final dataset shape: {final_df.shape}')
final_df.to_csv('data/solar_features.csv', index=False)
print('Saved → data/solar_features.csv')
final_df.head()

Final dataset shape: (2304, 15)
Saved → data/solar_features.csv


,voltage_mean,voltage_std,current_mean,current_std,power_mean,power_std,lux_mean,temperature_mean,humidity_mean,power_rate_of_change,voltage_rate_of_change,efficiency_ratio,fault_class,efficiency_score,maintenance_days
0,24.0,0.0,0.0,0.0,0.0,0.0,0.0,22.950,65.0,0.0,0.0,0.0,2,0.0,90
1,24.0,0.0,0.0,0.0,0.0,0.0,0.0,22.400,65.0,0.0,0.0,0.0,2,0.0,90
2,24.0,0.0,0.0,0.0,0.0,0.0,0.0,22.460,65.0,0.0,0.0,0.0,2,0.0,90
3,24.0,0.0,0.0,0.0,0.0,0.0,0.0,22.675,65.0,0.0,0.0,0.0,2,0.0,90
4,24.0,0.0,0.0,0.0,0.0,0.0,0.0,22.415,65.0,0.0,0.0,0.0,2,0.0,90


In [20]:
# Quick sanity checks
assert list(final_df.columns[:12]) == FEATURE_ORDER, 'Feature order mismatch!'
assert final_df['fault_class'].isin([0, 1, 2]).all(), 'Invalid fault_class values!'
assert final_df['efficiency_score'].between(0, 100).all(), 'efficiency_score out of range!'
assert final_df['maintenance_days'].between(0, 365).all(), 'maintenance_days out of range!'
print('All sanity checks passed ✓')

All sanity checks passed ✓
